# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Inspecting the tables on Hugging face for the most suitable one for my lane

In [15]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [16]:
from datasets import load_dataset_builder           # Import the function for inspecting a dataset's structure

# Load the dataset builder for the specified FlyRank warehouse table
builder = load_dataset_builder(
    "FlyRank/internship-warehouse",         # Hugging Face dataset repository
    "fact_content_daily_performance",       # Table to inspect
    token=HF_TOKEN                          # Hugging Face access token for authentication
)

# Display the dataset's features (column names and their data types)
print(builder.info.features)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

In [3]:
# Load the dataset builder for the dim_content table
builder = load_dataset_builder(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)
print(builder.info.features)

{'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'keyword_hash_id': Value('string'), 'url_hash_id': Value('string'), 'keyword_char_count': Value('int64'), 'keyword_token_count': Value('int64'), 'url_char_count': Value('int64'), 'content_created_date': Value('date32'), 'content_updated_date': Value('date32'), 'content_type': Value('string'), 'search_volume': Value('int64'), 'competition': Value('float64'), 'competition_level': Value('string'), 'cpc': Value('float64'), 'main_intent': Value('string'), 'backlinks': Value('int64'), 'category_count': Value('int64'), 'keyword_created_date': Value('date32'), 'provider_used': Value('string'), 'model_used': Value('string'), 'char_count': Value('int64'), 'word_count': Value('int64'), 'last_optimized_date': Value('date32'), 'optimization_eligible_date': Value('date32'), 'is_published': Value('bool'), 'is_deleted': Value('bool')}


In [4]:
# Load the dataset for the fact_content_query_90d table
builder = load_dataset_builder(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)

print(builder.info.features)

{'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'query_hash_id': Value('string'), 'query_char_count': Value('int64'), 'query_token_count': Value('int64'), 'window_start': Value('date32'), 'window_end': Value('date32'), 'impressions_90d': Value('int64'), 'clicks_90d': Value('int64'), 'impressions_last30': Value('int64'), 'clicks_last30': Value('int64'), 'impressions_prev30': Value('int64'), 'clicks_prev30': Value('int64'), 'avg_position_90d': Value('float64'), 'avg_position_last30': Value('float64'), 'avg_position_prev30': Value('float64'), 'content_total_impressions_90d': Value('int64'), 'content_visible_query_count': Value('int64'), 'rare_query_count': Value('int64'), 'rare_impressions_share': Value('float64'), 'anonymized_impressions_share': Value('float64')}


fact_content_daily_performance and dim_content tables contain useful signals for my lane: Refresh / Content Opportunity Scoring

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content item for one client.

For this analysis, I am using data from March 1, 2026 to March 31, 2026

In [6]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)


In [7]:
# Define the Hugging Face dataset path
rel = "hf://datasets/FlyRank/internship-warehouse"

# Execute a SQL query in DuckDB to count all rows
# in the fact_content_daily_performance table
con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [17]:
# Query the fact_content_daily_performance table using DuckDB,
# automatically read partition values from the file paths
# and return only the first 5 rows
con.sql(
    f"""
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    LIMIT 5
    """
)


┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [9]:
# Verify the proposed grain of the daily performance table.
# Expected grain: one row per report_date × client × content.

grain_check = con.sql(
    f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use March 2026 as the mid-panel development month.
    WHERE month = '2026-03'

    -- Group by the proposed unit of analysis.
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id

    -- Return any combinations that occur more than once.
    HAVING COUNT(*) > 1

    -- We only need to see whether duplicates exist.
    LIMIT 5
    """
)

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

In [10]:
# Verify the row count and date span of the March 2026 analysis slice.

march_summary = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use the mid-panel month selected for development.
    WHERE month = '2026-03'
    """
)

march_summary

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [11]:
# Check how many March 2026 rows have both GSC and GA4 data available.

availability_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS rows_with_both_available
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    """
)

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┐
│ rows_with_both_available │
│          int64           │
├──────────────────────────┤
│                   364347 │
└──────────────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.